# Lunar native environmental preprocessing

Ported from `notebooks/preprocessing/05_lunar_terrain_environment.ipynb`. Original source files are unchanged. Inputs and all generated data stay under `darrel stuff/data/`.

Uses the same seven SDEM region footprints, quality layers, six selected LOLA layers and ten illumination products as notebook 05. Its terrain-support calculations are retained because the original environmental EDA joins to those support views. Missing SDEM/LOLA inputs are not replaced by invented regions or alternate data. This is not CNN training.

**Code copied first; not executed locally.** No historical outputs are presented as new results. Install the original processing dependencies and read `../../EDA_Notes.md` before Run All.


Large native rasters are processed in bounded tiles; missingness, image-support thresholds and terrain methods are unchanged. Run this notebook alone if other Jupyter kernels are consuming memory.


In [1]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "outline.md").is_file() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository or one of its subfolders")
DATA_ROOT = ROOT / "darrel stuff/data"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "darrel stuff/EDA"))
from environment_workflow import LocalRun as Run, latest_completed_run, select_climate_manifest

print("Source data:", DATA_ROOT)
print("Processed outputs:", DATA_ROOT / "processed")


Source data: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data
Processed outputs: c:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed


In [2]:
import json
import numpy as np
import rasterio
from rasterio.crs import CRS
from rasterio.enums import Resampling
from notebooks.shared.processing import MOON_CRS, make_grid, roi_mask, write_raster, terrain
import terrain_raster_io
from terrain_raster_io import image_supported_grid, crop_native_windowed

SPACING_M = 30
ROUGHNESS_WINDOW = 11
MIN_IMAGE_SUPPORT_FRACTION = 0.99
sdem_root = DATA_ROOT / "moon/sdem/regions"
REGION_NAMES = sorted(p.name for p in sdem_root.iterdir() if p.is_dir())
# Set REGION_NAMES to one exact folder name for a first run.
regions = {}
for name in REGION_NAMES:
    elevation_paths = list((sdem_root / name).glob("*_GLDELEV_005.tif"))
    mask_paths = list((sdem_root / name).glob("*_GLDMASK_005.tif"))
    if len(elevation_paths) != 1 or len(mask_paths) != 1:
        raise ValueError(f"Missing/ambiguous elevation or image-count mask: {name}")
    with rasterio.open(elevation_paths[0]) as src:
        assert src.crs == CRS.from_string(MOON_CRS)
        regions[name] = {"elevation": elevation_paths[0], "mask": mask_paths[0], "bounds": tuple(src.bounds)}
    print(name, regions[name]["bounds"])
lola_root = DATA_ROOT / "moon/lola_product90/large_area_dem"
lola_names = ["LDEM_80S_40MPP_ADJ.TIF", "LDEM_80S_40MPP_ADJ_EFFRES.TIF", "LDEM_80S_40MPP_ADJ_ERR.TIF",
              "LDEC_80S_40MPP_ADJ.TIF", "LDSM_80S_40MPP_ADJ.TIF", "LDSM_80S_40MPP_ADJ_ERR.TIF"]
illumination = sorted(p for p in (DATA_ROOT / "moon/illumination_2016").rglob("*") if p.suffix.lower() == ".tif")
assert len(illumination) == 10 and all((lola_root / n).is_file() for n in lola_names)
assert SPACING_M >= 30 and 0 < MIN_IMAGE_SUPPORT_FRACTION <= 1


A3CLR22_10_Leibnitz_Beta_Plateau (65000.0, 110000.0, 81000.0, 126000.0)
A3CLR22_11_Malapert_Massif (-11000.0, 111000.0, 10000.0, 132000.0)
A3CLR22_12_deGerlache_Kocher_Massif (-123400.0, -66100.0, -103400.0, -46100.0)
A3CLR22_2_Haworth (-52900.0, 75600.0, -23100.0, 105400.0)
A3CLR22_3_Faustini_Rim_A (54000.0, -10000.0, 74000.0, 10000.0)
A3CLR22_4_deGerlache_Rim_2 (-58500.0, 13000.0, -38500.0, 33000.0)
A3CLR22_5_Connecting_Ridge_Extension (-38700.0, -16100.0, -18700.0, 3900.0)


In [3]:
config = {"regions": REGION_NAMES, "spacing_m": SPACING_M, "roughness_window": ROUGHNESS_WINDOW,
          "minimum_image_support_fraction": MIN_IMAGE_SUPPORT_FRACTION, "moon_radius_m": 1737400,
          "image_count_alignment": "nearest to elevation grid; missing coverage unsupported",
          "elevation_aggregation": "average supported source cells", "environment": "native grid crops, scaled values",
          "native_io_tile_size": 512, "gdal_cache_mb": 32}
with rasterio.Env(GDAL_CACHEMAX=32*1024**2), Run(ROOT, "lunar_terrain_environment", config, "Moon/Terrain_And_Soil/05_lunar_terrain_environment.ipynb") as run:
    run.input(Path(terrain_raster_io.__file__))
    summaries = {}
    for name, info in regions.items():
        run.input(info["elevation"])
        run.input(info["mask"])
        run.input(sdem_root / name / "README.txt")
        grid = make_grid(info["bounds"], MOON_CRS, SPACING_M)
        # Bound native I/O in 512-pixel tiles; keep the same nearest-count support,
        # average aggregation, 30 m lattice and 0.99 acceptance threshold.
        elevation, fraction = image_supported_grid(
            info["elevation"], info["mask"], grid, run.directory)
        inside = roi_mask(grid, info["bounds"], MOON_CRS)
        elevation[~inside] = np.nan
        fraction[~inside] = np.nan
        elevation[fraction < MIN_IMAGE_SUPPORT_FRACTION] = np.nan
        folder = f"moon/{name}/grid_{SPACING_M}m"
        write_raster(run.path(f"{folder}/elevation_m.tif"), elevation, grid, "m", vertical_reference="radius 1737400 m")
        write_raster(run.path(f"{folder}/image_support_fraction.tif"), fraction, grid, "fraction")
        for feature, values in terrain(elevation, SPACING_M, ROUGHNESS_WINDOW).items():
            units = "degree" if feature == "slope_deg" else "1/m" if feature == "laplacian_per_m" else "m"
            write_raster(run.path(f"{folder}/{feature}.tif"), values, grid, units)
        native = []
        # Preserve source-defined image counts, uncertainty, best resolution and supplied terrain metrics.
        quality = [p for p in (sdem_root / name).rglob("*.tif")
                   if any(token in p.name for token in ("GLDMASK", "GLDSIGM", "GLDISGM", "GLDBRES", "GLDSBCT", "GLDTSLP", "GLDTRGH"))]
        for source in quality + [lola_root / n for n in lola_names] + illumination:
            run.input(source)
            category = "sdem_quality" if source.is_relative_to(sdem_root) else "lola" if source.parent == lola_root else "illumination"
            output = run.path(f"moon/{name}/native/{category}/{source.name}")
            # The region README documents the CRS omitted from slope/roughness GeoTIFFs.
            documented_sdem = category == "sdem_quality" and source.parent.name in ("SLOPE_MAPS", "ROUGHNESS_MAPS")
            metadata = crop_native_windowed(source, output, info["bounds"], MOON_CRS,
                                   fallback_crs=MOON_CRS if documented_sdem else None,
                                   crs_evidence=str(sdem_root / name / "README.txt") if documented_sdem else None)
            native.append(metadata)
        summaries[name] = {"bounds_m": info["bounds"], "valid_elevation_fraction": float(np.isfinite(elevation).mean()),
                           "native_layers": native}
        run.path(f"moon/{name}/quality.json").write_text(json.dumps(summaries[name], indent=2), encoding="utf-8")
        print(name, "valid elevation fraction", summaries[name]["valid_elevation_fraction"])
    run.record["summaries"] = summaries
print("Completed:", run.directory)


A3CLR22_10_Leibnitz_Beta_Plateau valid elevation fraction 0.9316619674844646
A3CLR22_11_Malapert_Massif valid elevation fraction 0.9736845323007948
A3CLR22_12_deGerlache_Kocher_Massif valid elevation fraction 0.9574518627415827
A3CLR22_2_Haworth valid elevation fraction 0.9125952900501602
A3CLR22_3_Faustini_Rim_A valid elevation fraction 0.7500448877357728
A3CLR22_4_deGerlache_Rim_2 valid elevation fraction 0.9201194005695803
A3CLR22_5_Connecting_Ridge_Extension valid elevation fraction 0.9520262357576834
Completed: C:\Darrel\Self-Projects\1 group proj\NASA-Space-Apps-Challenge-2026\darrel stuff\data\processed\lunar_terrain_environment\20261109T153250161318Z_0ecb9f


## Preserve native environmental semantics

Source nodata is masked before scale/offset decoding. Crops keep each source's native grid. Unknown units remain unconfirmed; do not assume sky visibility is an illumination fraction. Image-support exclusions are not missing source elevation. Mean illumination does not establish darkness duration.